## Feature selection

#### 5 classes

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
from tensorflow.keras import regularizers
from tensorflow.keras.losses import CategoricalCrossentropy


era5 = xr.open_dataset(r"D:\\Documents\\thesis\\processed_data\\era5-land-all-variables\\era5land_ebro_daily_mean3.nc")
df = era5.to_dataframe()

df['day_of_year'] = df.index.dayofyear

# Ensure time column is datetime and extract the month
df['time'] = pd.to_datetime(df.index)
df['month'] = df['time'].dt.month
# This helps the model understand that Dec and Jan are neighbors
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Calculate Mean and Std Dev per month (and per pixel/station if applicable)
stats = df.groupby('month')['swvl1'].agg(['mean', 'std']).reset_index()

# Merge stats back to original dataframe
df = df.merge(stats, on='month')

# Calculate Z-score
df['z_score'] = (df['swvl1'] - df['mean']) / df['std']

# Create Buckets
bins = [-float('inf'), -1.5, -0.5, 0.5, 1.5, float('inf')]
labels = ['Severely Dry', 'Moderately Dry', 'Normal', 'Moderately Wet', 'Severely Wet']
labels = [0, 1, 2, 3, 4]
df['category'] = pd.cut(df['z_score'], bins=bins, labels=labels).astype(int)
df["target_next_month"] = df["category"].shift(-30)
df['tp_30d_sum'] = df['tp'].rolling(window=30, center=False).sum()
df['tp_90d_sum'] = df['tp'].rolling(window=90, center=False).sum()
df = df.dropna()

features_to_drop = ['z_score', 'target_next_month', 'time', 'mean'] 

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

def run_feature_selection(df, lead_time=30):
    # Clean up columns
    # We want to keep the original features, but drop targets/time
    cols_to_exclude = ['category', 'target_next_month', 'time']
    X = df.drop(columns=[c for c in cols_to_exclude if c in df.columns])
    
    # Create the shifted target manually (no sequences needed for XGBoost)
    # We want to predict the category 'lead_time' days into the future
    y = df['category'].shift(-lead_time)
    
    # Remove the last 'lead_time' rows where target is now NaN
    X = X.iloc[:-lead_time]
    y = y.iloc[:-lead_time]
    
    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    
    # Initialize and Train XGBoost
    # We use a shallow tree (max_depth=3 or 5) to prevent it from picking up on 'accidental' noise
    model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
    
    print(f"Analyzing feature importance for {lead_time}-day lead time...")
    model.fit(X_train, y_train)
    
    # Extract and Plot Importances
    importances = pd.DataFrame({
        'Feature': X.columns,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    
    print(importances.head(15))
    
    return importances

df_reduced = df.drop(columns=features_to_drop)
feature_results = run_feature_selection(df_reduced, lead_time=7)

top_15 = feature_results['Feature'].head(15).tolist()
print("\nTop 15 Features recommended for your LSTM:")
print(top_15)

feature_results = run_feature_selection(df_reduced, lead_time=14)

top_15 = feature_results['Feature'].head(15).tolist()
print("\nTop 15 Features recommended for your LSTM:")
print(top_15)

feature_results = run_feature_selection(df_reduced, lead_time=21)

top_15 = feature_results['Feature'].head(15).tolist()
print("\nTop 15 Features recommended for your LSTM:")
print(top_15)

feature_results = run_feature_selection(df_reduced, lead_time=30)

top_15 = feature_results['Feature'].head(15).tolist()
print("\nTop 15 Features recommended for your LSTM:")
print(top_15)

Analyzing feature importance for 7-day lead time...


c:\Users\albat\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


        Feature  Importance
20        swvl1    0.131841
11         ssrd    0.071284
9          slhf    0.071101
29    month_cos    0.057049
31   tp_30d_sum    0.045875
21        swvl2    0.041141
6           sro    0.040590
30          std    0.030867
17         stl1    0.030366
2             e    0.029670
26  day_of_year    0.028709
32   tp_90d_sum    0.027977
28    month_sin    0.027665
22        swvl3    0.026783
27        month    0.026337

Top 15 Features recommended for your LSTM:
['swvl1', 'ssrd', 'slhf', 'month_cos', 'tp_30d_sum', 'swvl2', 'sro', 'std', 'stl1', 'e', 'day_of_year', 'tp_90d_sum', 'month_sin', 'swvl3', 'month']
Analyzing feature importance for 14-day lead time...


c:\Users\albat\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


        Feature  Importance
20        swvl1    0.081441
29    month_cos    0.081276
27        month    0.056409
21        swvl2    0.055246
30          std    0.045021
28    month_sin    0.043657
2             e    0.040933
11         ssrd    0.039889
23        swvl4    0.037717
31   tp_30d_sum    0.036691
7          ssro    0.035092
26  day_of_year    0.034876
32   tp_90d_sum    0.033007
22        swvl3    0.031178
19           sd    0.028278

Top 15 Features recommended for your LSTM:
['swvl1', 'month_cos', 'month', 'swvl2', 'std', 'month_sin', 'e', 'ssrd', 'swvl4', 'tp_30d_sum', 'ssro', 'day_of_year', 'tp_90d_sum', 'swvl3', 'sd']
Analyzing feature importance for 21-day lead time...


c:\Users\albat\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


        Feature  Importance
29    month_cos    0.169439
21        swvl2    0.069559
28    month_sin    0.044800
27        month    0.044752
30          std    0.042069
23        swvl4    0.040167
7          ssro    0.037797
31   tp_30d_sum    0.036737
32   tp_90d_sum    0.034413
26  day_of_year    0.034004
22        swvl3    0.033869
20        swvl1    0.033034
18         stl2    0.028345
11         ssrd    0.025129
19           sd    0.025086

Top 15 Features recommended for your LSTM:
['month_cos', 'swvl2', 'month_sin', 'month', 'std', 'swvl4', 'ssro', 'tp_30d_sum', 'tp_90d_sum', 'day_of_year', 'swvl3', 'swvl1', 'stl2', 'ssrd', 'sd']
Analyzing feature importance for 30-day lead time...


c:\Users\albat\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


        Feature  Importance
29    month_cos    0.126011
28    month_sin    0.054775
27        month    0.052664
20        swvl1    0.043479
23        swvl4    0.042905
22        swvl3    0.041624
21        swvl2    0.041107
31   tp_30d_sum    0.040448
7          ssro    0.039833
11         ssrd    0.038451
32   tp_90d_sum    0.038424
30          std    0.037173
26  day_of_year    0.036113
18         stl2    0.027305
10          ssr    0.026050

Top 15 Features recommended for your LSTM:
['month_cos', 'month_sin', 'month', 'swvl1', 'swvl4', 'swvl3', 'swvl2', 'tp_30d_sum', 'ssro', 'ssrd', 'tp_90d_sum', 'std', 'day_of_year', 'stl2', 'ssr']


#### 3 classes

In [ ]:
era5 = xr.open_dataset(r"D:\\Documents\\thesis\\processed_data\\era5-land-all-variables\\era5land_ebro_daily_mean3.nc")
df = era5.to_dataframe()

df['day_of_year'] = df.index.dayofyear

# Ensure time column is datetime and extract the month
df['time'] = pd.to_datetime(df.index)
df['month'] = df['time'].dt.month
# This helps the model understand that Dec and Jan are neighbors
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Calculate Mean and Std Dev per month (and per pixel/station if applicable)
stats = df.groupby('month')['swvl1'].agg(['mean', 'std']).reset_index()

# Merge stats back to original dataframe
df = df.merge(stats, on='month')

# Calculate Z-score
df['z_score'] = (df['swvl1'] - df['mean']) / df['std']

# Create Buckets
bins = [-float('inf'), -0.5, 0.5, float('inf')]
labels = ['Dry', 'Normal', 'Wet']
labels = [0, 1, 2]
df['category'] = pd.cut(df['z_score'], bins=bins, labels=labels).astype(int)
df["target_next_month"] = df["category"].shift(-30)
df['tp_30d_sum'] = df['tp'].rolling(window=30, center=False).sum()
df['tp_90d_sum'] = df['tp'].rolling(window=90, center=False).sum()
df = df.dropna()

# Add date column's exact name to this list
features_to_drop = ['z_score', 'target_next_month', 'time', 'mean'] 

df_reduced = df.drop(columns=features_to_drop)
feature_results = run_feature_selection(df_reduced, lead_time=7)

top_15 = feature_results['Feature'].head(15).tolist()
print("\nTop 15 Features recommended for your LSTM:")
print(top_15)

feature_results = run_feature_selection(df_reduced, lead_time=14)

top_15 = feature_results['Feature'].head(15).tolist()
print("\nTop 15 Features recommended for your LSTM:")
print(top_15)

feature_results = run_feature_selection(df_reduced, lead_time=21)

top_15 = feature_results['Feature'].head(15).tolist()
print("\nTop 15 Features recommended for your LSTM:")
print(top_15)

feature_results = run_feature_selection(df_reduced, lead_time=30)

top_15 = feature_results['Feature'].head(15).tolist()
print("\nTop 15 Features recommended for your LSTM:")
print(top_15)

Analyzing feature importance for 7-day lead time...


c:\Users\albat\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


        Feature  Importance
20        swvl1    0.178176
6           sro    0.084257
29    month_cos    0.044860
21        swvl2    0.040940
31   tp_30d_sum    0.038497
30          std    0.035942
15          d2m    0.034331
26  day_of_year    0.032695
32   tp_90d_sum    0.030925
22        swvl3    0.029389
27        month    0.028343
23        swvl4    0.027054
3         evabs    0.026844
18         stl2    0.026669
7          ssro    0.026400

Top 15 Features recommended for your LSTM:
['swvl1', 'sro', 'month_cos', 'swvl2', 'tp_30d_sum', 'std', 'd2m', 'day_of_year', 'tp_90d_sum', 'swvl3', 'month', 'swvl4', 'evabs', 'stl2', 'ssro']
Analyzing feature importance for 14-day lead time...


c:\Users\albat\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


        Feature  Importance
20        swvl1    0.117700
27        month    0.061501
21        swvl2    0.051041
28    month_sin    0.045713
30          std    0.042688
23        swvl4    0.042406
31   tp_30d_sum    0.041872
29    month_cos    0.039700
32   tp_90d_sum    0.038532
22        swvl3    0.037907
26  day_of_year    0.035651
7          ssro    0.033846
16          skt    0.031081
18         stl2    0.030902
6           sro    0.028084

Top 15 Features recommended for your LSTM:
['swvl1', 'month', 'swvl2', 'month_sin', 'std', 'swvl4', 'tp_30d_sum', 'month_cos', 'tp_90d_sum', 'swvl3', 'day_of_year', 'ssro', 'skt', 'stl2', 'sro']
Analyzing feature importance for 21-day lead time...


c:\Users\albat\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


        Feature  Importance
21        swvl2    0.067250
20        swvl1    0.063632
27        month    0.052097
30          std    0.051094
28    month_sin    0.045043
23        swvl4    0.044165
29    month_cos    0.041101
31   tp_30d_sum    0.040356
22        swvl3    0.039226
9          slhf    0.038948
26  day_of_year    0.037744
7          ssro    0.037716
32   tp_90d_sum    0.036399
16          skt    0.035624
18         stl2    0.033963

Top 15 Features recommended for your LSTM:
['swvl2', 'swvl1', 'month', 'std', 'month_sin', 'swvl4', 'month_cos', 'tp_30d_sum', 'swvl3', 'slhf', 'day_of_year', 'ssro', 'tp_90d_sum', 'skt', 'stl2']
Analyzing feature importance for 30-day lead time...


c:\Users\albat\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


        Feature  Importance
20        swvl1    0.063645
29    month_cos    0.053068
23        swvl4    0.049282
21        swvl2    0.046939
7          ssro    0.046475
30          std    0.044692
22        swvl3    0.044178
31   tp_30d_sum    0.043691
28    month_sin    0.043454
32   tp_90d_sum    0.042015
26  day_of_year    0.041280
18         stl2    0.038703
27        month    0.037742
17         stl1    0.030857
11         ssrd    0.029821

Top 15 Features recommended for your LSTM:
['swvl1', 'month_cos', 'swvl4', 'swvl2', 'ssro', 'std', 'swvl3', 'tp_30d_sum', 'month_sin', 'tp_90d_sum', 'day_of_year', 'stl2', 'month', 'stl1', 'ssrd']


In [ ]:
top_features = ['swvl1', 'sro', 'std', 'slhf', 'month_cos', 'd2m','swvl3', 'e', 'stl1', 'tp_30d_sum', 'category']